# Lynchburg：12 个直线街景候选点
使用 CPU。将 GOOGLE_MAPS_API_KEY 添加到 Colab Secrets 并启用 Notebook access。
保留已经确认的 A/B/C，再从 C 沿 324° 每 35 米外推，候选范围约 394 米。外推不等于已确认路线；遇到异常停止，不跳过缺口。
每次运行最多 12 次元数据查询、12 次图片请求；重复运行重新请求。这里只显示街景并下载无密钥坐标报告，不运行生成模型。
拍摄日期仅用于记录，不作为拒绝条件。允许不同月份、季节和光照的街景。

来源筛选已改为 source=google，仅查询 Google 官方采集；仍需检查实际道路视角。


In [ ]:
import math, json
import requests
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from google.colab import userdata, files

# CPU only. Exactly 12 candidate metadata lookups; at most 12 image requests.
# No retries, no GPU calls, no key or request URLs in exported data.
A = (37.41160054638993, -79.1393862038534)
B = (37.41187948911811, -79.13965684087397)
C = (37.41216824976726, -79.1399143846171)
HEADING, FOV, PITCH = 324, 90, 0
BASE = "https://maps.googleapis.com/maps/api/streetview"

def destination(p, meters, heading):
    lat, lon, h = map(math.radians, (*p, heading))
    d = meters / 6371000
    lat2 = math.asin(math.sin(lat)*math.cos(d) + math.cos(lat)*math.sin(d)*math.cos(h))
    lon2 = lon + math.atan2(math.sin(h)*math.sin(d)*math.cos(lat), math.cos(d)-math.sin(lat)*math.sin(lat2))
    return math.degrees(lat2), math.degrees(lon2)

def distance(a, b):
    p, q = math.radians(a[0]), math.radians(b[0])
    h = math.sin((q-p)/2)**2 + math.cos(p)*math.cos(q)*math.sin(math.radians(b[1]-a[1])/2)**2
    return 12742000*math.asin(min(1, math.sqrt(h)))

def bearing(a, b):
    p, q = math.radians(a[0]), math.radians(b[0])
    d = math.radians(b[1]-a[1])
    return math.degrees(math.atan2(math.sin(d)*math.cos(q), math.cos(p)*math.sin(q)-math.sin(p)*math.cos(q)*math.cos(d))) % 360

def run():
    try:
        key = userdata.get("GOOGLE_MAPS_API_KEY").strip()
    except Exception:
        print("无法读取 Secret，请检查名称和 Notebook access。")
        return
    targets = [A, B, C] + [destination(C, 35*i, HEADING) for i in range(1, 10)]
    rows, seen, previous, cumulative = [], set(), None, 0.0
    session = requests.Session()
    print("检查 12 个候选位置；后三百多米是直线外推，尚未确认道路连续性。")
    for i, target in enumerate(targets):
        row = {"index": i+1, "target_lat": target[0], "target_lng": target[1]}
        try:
            r = session.get(BASE + "/metadata", params={"location": f"{target[0]},{target[1]}", "radius": 12, "source": "google", "key": key}, timeout=25)
            if r.status_code != 200:
                row["status"] = f"HTTP_{r.status_code}"
            else:
                m = r.json()
                row["status"] = m.get("status", "UNKNOWN")
                if row["status"] == "OK":
                    actual = (m["location"]["lat"], m["location"]["lng"])
                    pano = m["pano_id"]
                    offset = distance(target, actual)
                    gap = distance(previous, actual) if previous else 0
                    direction = bearing(previous, actual) if previous else HEADING
                    delta = abs((direction-HEADING+180)%360-180)
                    row.update(pano_id=pano, lat=actual[0], lng=actual[1], date=m.get("date", "unknown"), offset_m=round(offset, 1), gap_m=round(gap, 1), bearing_deg=round(direction, 1))
                    if pano in seen:
                        row["status"] = "REJECT_DUPLICATE"
                    elif offset > 12:
                        row["status"] = "REJECT_OFFSET"
                    elif previous and (gap < 15 or gap > 55 or delta > 15):
                        row["status"] = "REJECT_ROUTE_GEOMETRY"
                    else:
                        seen.add(pano)
                        cumulative += gap
                        row["distance_m"] = round(cumulative, 1)
                        previous = actual
        except (requests.RequestException, ValueError, KeyError):
            row["status"] = "REQUEST_OR_RESPONSE_ERROR"
        rows.append(row)
        print(f"Point {i+1:02d}: {row['status']}")
        # Do not silently bridge a missing/invalid point into a new route.
        if row["status"] != "OK":
            print("停止向前外推：需要检查此处道路或覆盖情况。")
            break
    display(pd.DataFrame([{k:v for k,v in r.items() if k != "pano_id"} for r in rows]))
    good = [r for r in rows if r["status"] == "OK"]
    print(f"通过坐标筛选 {len(good)} 个点；仍须看图确认同一条街。")
    if good:
        fig, axes = plt.subplots(math.ceil(len(good)/2), 2, figsize=(16, 5*math.ceil(len(good)/2)), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for ax, row in zip(axes.flat, good):
            ax.set_title(f"P{row['index']:02d} | {row['distance_m']} m | {row['date']} | 324 deg")
            try:
                r = session.get(BASE, params={"pano": row["pano_id"], "size": "640x360", "heading": HEADING, "pitch": PITCH, "fov": FOV, "return_error_code": "true", "key": key}, timeout=25)
                row["image_http"] = r.status_code
                if r.status_code == 200 and r.headers.get("Content-Type", "").startswith("image/"):
                    ax.imshow(Image.open(BytesIO(r.content)))
                    row["image_ok"] = True
                else:
                    row["image_ok"] = False
                    ax.text(.1, .5, f"HTTP {r.status_code}", transform=ax.transAxes)
            except Exception:
                row["image_ok"] = False
                ax.text(.1, .5, "Image request failed", transform=ax.transAxes)
        plt.tight_layout()
        plt.show()
    report = {"purpose": "candidate route inspection, not generation-ready approval", "visual_route_verified": False, "heading": HEADING, "fov": FOV, "pitch": PITCH, "size": [640,360], "rows": rows}
    with open("lynchburg_route_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print("下载坐标检查报告（不含密钥和图片）。请上传报告并发路线对比图。")
    files.download("lynchburg_route_report.json")

try:
    run()
except Exception:
    print("执行未完成；请发送已有输出，不要发送密钥。")
